In [12]:
import json
import sys
import os
import time
import ast
from tqdm import tqdm
from openai import OpenAI
from striprtf.striprtf import rtf_to_text
from dotenv import load_dotenv
import importlib
import csv

# Load environment variables from current directory .env and override existing ones
load_dotenv(".env", override=True)

# load api key from environment
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

# Get the model from environment variable
model = 'gpt-4.1-mini-2025-04-14'

# Get temperature and max_tokens from environment variables with defaults
temperature = float(os.getenv("OPENAI_TEMPERATURE", "0.7"))
max_tokens = 4096

#set client with key from .env
client = OpenAI(api_key=api_key)

In [3]:
#Function for sending batch job, waiting on it, and returning the results
def manage_batch_job(batch_items):

    output_filename = "batch_request.jsonl"

    with open(output_filename, 'w') as f:
        for item in batch_items:
            json.dump(item, f)
            f.write('\n')


    batch_file = output_filename
    batch_input_file = client.files.create(
    file=open(batch_file, "rb"),
    purpose="batch"
    )
    batch_input_file_id = batch_input_file.id

    batch_job = client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
        "description": "Batch relevance scoring"
    }
    )


    while True:
        batch_job = client.batches.retrieve(batch_job.id)

        if batch_job.status == "failed":
            print(f"Batch job failed {batch_job.id}, exiting...")
            break
        elif batch_job.status == "running":
            print(f"Batch job {batch_job.id} is still running...")
            time.sleep(10)
        elif batch_job.status != "completed":
            print(f"Batch job {batch_job.id} is currently: {batch_job.status}. Waiting...")
            time.sleep(10) # Wait for a few seconds before checking again
        else:
            print(f"Batch job {batch_job.id} is done.")
            break

    # Download batch response

    batch = client.batches.retrieve(batch_job.id)

    batch_output = batch.output_file_id


    # Load batch response
    batch_response = client.files.content(batch_output)
    output_string = batch_response.read().decode("utf-8")   # Decode to string


    output_data = read_jsonl_from_string(output_string)

    return output_data


In [4]:
def retrieve_batch(batch_job_id):
    batch = client.batches.retrieve(batch_job_id)

    batch_output = batch.output_file_id


    # Load batch response
    batch_response = client.files.content(batch_output)
    output_string = batch_response.read().decode("utf-8")   # Decode to string


    output_data = read_jsonl_from_string(output_string)

    return output_data


In [5]:
#Function for reading in the results from batch requests

def read_jsonl_from_string(jsonl_string):
    results = []
    for line in jsonl_string.splitlines():
        if line.strip():
            try:
                results.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON: {e}")
    return results

In [6]:
def gpt_generate(messages, model=model, temperature=temperature, max_tokens=max_tokens):

    max_retries = 5
    retries = 0

    while True:
        try:
            response = client.chat.completions.create(
                model=model, messages=messages, temperature=temperature, max_tokens=max_tokens
            )
            return response.choices[0].message.content
        except Exception as e:
            print(f"Error: {e}")
            retries += 1
            if retries >= max_retries:
                print("Max retries reached. Exiting.")
                sys.exit(1)
            wait_time = 2 ** retries
            print(f"Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
            continue

Load your original prompt_dataset from a file. Will need to have a 'prompt' item and some form of preference annotation for each prompt. 

In [7]:
prompt_dataset_filepath = r'C:\Users\hbbel\OneDrive\Desktop\Research\Git Repos\Principle-Elicitation\Princ_from_reasons\data_subset.json'

with open(prompt_dataset_filepath, 'r') as f:
    prompt_dataset = json.load(f)








Load the candidate principles. 

In [8]:
principles_filepath = r'C:\Users\hbbel\OneDrive\Desktop\Research\Git Repos\Principle-Elicitation\Principle Generations\12_12_FINAL_generations\12_12_FINAL_summarized_principles.json'

with open(principles_filepath, 'r') as f:
    principles = json.load(f)

print(len(principles))



622


Load prompts

In [56]:
parallelized_system_prompt_path = r'C:\Users\hbbel\OneDrive\Desktop\Research\Git Repos\Principle-Elicitation\Principle Scoring\Prompts\parallelized_system_prompt.txt'
with open(parallelized_system_prompt_path, 'r') as f:
    parallelized_system_prompt = f.read()

parallelized_template_path = r'C:\Users\hbbel\OneDrive\Desktop\Research\Git Repos\Principle-Elicitation\Principle Scoring\Prompts\parallelized_template.txt'
with open(parallelized_template_path, 'r') as f:
    parallelized_template = f.read()



Calculate relevance scores for each principle + prompt combo.

For every prompt, check the relevance of every principle. Score 1-5.

Uses batch API, so run & check back in 5-10 min

In [57]:
prompt_template = parallelized_template
prompt_id = 0
full_batch_request = []

principles_per_batch = 15


for dataset_prompt in prompt_dataset:
    i = 0
    j = 0
    test_item = dataset_prompt


    test_prompt = prompt_dataset[0]['prompt']

    test_response_1 = prompt_dataset[0]['response_1']

    test_response_2 = prompt_dataset[0]['response_2']

    for principle in principles:

        target = "Principle " + str(i)+": " + principle['summarized_principle']
        prompt_template += "\n" + target

        i += 1
        j += 1

        if j >= principles_per_batch:
            prompt_template = prompt_template.replace("{PROMPT}", test_prompt)
            prompt_template = prompt_template.replace("{RESPONSE_1}", test_response_1)
            prompt_template = prompt_template.replace("{RESPONSE_2}", test_response_2)


            messages = [
                {"role": "system", "content": parallelized_system_prompt},
                {"role": "user", "content": prompt_template}
            ]

            batch_request = {"custom_id": f'{prompt_id}_{i-principles_per_batch}_{i-1}', "method": "POST", "url": "/v1/chat/completions", "body": {"model": model,
                            "messages": messages, "max_tokens": 4080, "temperature": 0}}

            full_batch_request.append(batch_request)
            j = 0
        
            prompt_template = parallelized_template

    # Catch any remaining principles after loop before resetting and moving to next prompt
    prompt_template = prompt_template.replace("{PROMPT}", test_prompt)
    prompt_template = prompt_template.replace("{RESPONSE_1}", test_response_1)
    prompt_template = prompt_template.replace("{RESPONSE_2}", test_response_2)


    #ID format is promptID_start principle Index_end principle Index

    messages = [
                {"role": "system", "content": parallelized_system_prompt},
                {"role": "user", "content": prompt_template}
            ]

    batch_request = {"custom_id": f'{prompt_id}_{i-principles_per_batch}_{i-1}', "method": "POST", "url": "/v1/chat/completions", "body": {"model": model,
                    "messages": messages, "max_tokens": max_tokens, "temperature": 0}}

    full_batch_request.append(batch_request)

    j = 0
    prompt_template = parallelized_template

    prompt_id += 1


print(f"Total batch requests: {len(full_batch_request)}")


Total batch requests: 29400


In [58]:
manage_batch_job(full_batch_request)

Batch job batch_694319b666b08190b72d0079761c57c9 is currently: validating. Waiting...
Batch job batch_694319b666b08190b72d0079761c57c9 is currently: validating. Waiting...
Batch job batch_694319b666b08190b72d0079761c57c9 is currently: validating. Waiting...
Batch job batch_694319b666b08190b72d0079761c57c9 is currently: validating. Waiting...
Batch job batch_694319b666b08190b72d0079761c57c9 is currently: validating. Waiting...
Batch job batch_694319b666b08190b72d0079761c57c9 is currently: validating. Waiting...
Batch job batch_694319b666b08190b72d0079761c57c9 is currently: validating. Waiting...
Batch job batch_694319b666b08190b72d0079761c57c9 is currently: in_progress. Waiting...
Batch job batch_694319b666b08190b72d0079761c57c9 is currently: in_progress. Waiting...
Batch job batch_694319b666b08190b72d0079761c57c9 is currently: in_progress. Waiting...
Batch job batch_694319b666b08190b72d0079761c57c9 is currently: in_progress. Waiting...
Batch job batch_694319b666b08190b72d0079761c57c9 i

APIConnectionError: Connection error.

In [69]:
out_data = retrieve_batch('batch_6942d3b53b888190bb35945398572bbd')

with open('parallelized_scoring_results_3.jsonl', 'w') as f:
    for item in out_data:
        json.dump(item, f)
        f.write('\n')

In [60]:
possible_strengths = {}
total_annotations = 0
for prompt in prompt_dataset:
    for annotator in prompt['all_preferences_unprocessed']:
        total_annotations += 1
        strength = annotator['strength']
        if strength not in possible_strengths:
            possible_strengths[strength] = 0

print(possible_strengths)
print(total_annotations)

{-1: 0, 1: 0, 3: 0, 2: 0, -2: 0, -3: 0}
2359


In [70]:
aggregate_scores = {}
num_irrelevant = {}
total_irrelevant = 0

total_correct = 0

x = 0

for item in out_data:

    custom_id = item['custom_id']  

    # split custom_id to get prompt and principle indices
    prompt_id, start_idx, end_idx = custom_id.split('_')
    start_idx = int(start_idx)
    end_idx = int(end_idx)
    prompt_id = int(prompt_id)

    # Pull original annotations from dataset
    original_dataset_annotations = prompt_dataset[prompt_id]['all_preferences_unprocessed'] 


    # Extract the model response, and read as a dictionary
    score_results = item['response']['body']['choices'][0]['message']['content']
    score_dict = ast.literal_eval(score_results)


    for principle, score in score_dict.items():

        # Add principle to principle dict if not already present
        if principle not in aggregate_scores:
            aggregate_scores[principle] = 0
            


        score = int(score) if isinstance(score, str) and score.isdigit() else score
        for annotation in original_dataset_annotations:
            if annotation['strength'] < 0 and score == 1:
                aggregate_scores[principle] = aggregate_scores.get(principle, 0) + 1
                total_correct += 1
            elif annotation['strength'] > 0 and score == 2:
                aggregate_scores[principle] = aggregate_scores.get(principle, 0) + 1
                total_correct += 1
            elif score == 'Not Relevant':
                num_irrelevant[principle] = num_irrelevant.get(principle, 0) + 1
                total_irrelevant += 1


print(len(aggregate_scores.keys()))
print(aggregate_scores)
print(len(num_irrelevant.keys()))
print(num_irrelevant)
print(f"Total correct annotations: {total_correct}")
print(f"Total irrelevant annotations: {total_irrelevant}")

# Now we can calculate the score for each principle
# Score = (Number correct annotations) / ((total annotations - number  irrelevant) + 9)

final_scores = {}
total = total_annotations

for principle in aggregate_scores.keys():
    correct = aggregate_scores[principle]
    irrelevant = num_irrelevant.get(principle, 0)
    score = correct / ((total - irrelevant) + 9)
    final_scores[principle] = score

# Sort final scores, print top 20 principles
sorted_final_scores = dict(sorted(final_scores.items(), key=lambda item: item[1], reverse=True))

print("Top 20 Principles by Score:")
for i, (principle, score) in enumerate(sorted_final_scores.items()):
    if i < 20:
        
        # Get the principle id (number value after 'principle_')
        principle_id = principle.split('_')[1]

        print(f"ID: {principle_id}: {score} - {principles[int(principle_id)]['summarized_principle']}")
    else:
        break

622
{'principle_0': 0, 'principle_1': 314, 'principle_2': 0, 'principle_3': 0, 'principle_4': 1306, 'principle_5': 0, 'principle_6': 1058, 'principle_7': 0, 'principle_8': 1296, 'principle_9': 1296, 'principle_10': 1303, 'principle_11': 0, 'principle_12': 0, 'principle_13': 18, 'principle_14': 0, 'principle_15': 1053, 'principle_16': 1306, 'principle_17': 1298, 'principle_18': 1170, 'principle_19': 0, 'principle_20': 167, 'principle_21': 1278, 'principle_22': 1199, 'principle_23': 0, 'principle_24': 0, 'principle_25': 1050, 'principle_26': 0, 'principle_27': 0, 'principle_28': 0, 'principle_29': 1101, 'principle_30': 1021, 'principle_31': 1008, 'principle_32': 0, 'principle_33': 0, 'principle_34': 1306, 'principle_35': 10, 'principle_36': 0, 'principle_37': 1060, 'principle_38': 0, 'principle_39': 0, 'principle_40': 1092, 'principle_41': 1053, 'principle_42': 1054, 'principle_43': 0, 'principle_44': 1306, 'principle_45': 1053, 'principle_46': 1307, 'principle_47': 1307, 'principle_48':